# Episodic Memory with Write-Time Reflection

This notebook demonstrates `ReflectiveMemory` and `JSONMemoryStore` working together
to give an `LLMAgent` episodic memory that distils a short lesson from each completed
task at write time.

Two properties are illustrated:

- **Write-time reflection.** After each task the agent records an episode.
  Before the episode is stored, `ReflectiveMemory` calls the LLM to produce a
  one-sentence lesson from the task and its outcome. That lesson is attached to the
  episode under `additional_data["reflection"]` and rendered automatically inside a
  `<reflection>` tag whenever the episode is displayed.
- **Recalled lessons.** When a new task arrives, `recall()` surfaces the three most
  recent episodes. Because each episode now carries a pre-distilled lesson, the agent
  sees not only what happened but the key takeaway from it. For synthesis tasks that
  draw on prior research, the recalled reflections surface the essential facts
  directly rather than requiring the agent to re-parse full result transcripts.

> **This notebook uses the same PokéAPI tool as `recency_memory.ipynb`.**
> Reading that notebook first is recommended.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code in this notebook you need Ollama installed locally with its
service running. Download Ollama from https://ollama.com/download, then start the
service by running `ollama serve` in a terminal.

## Setup

In [6]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

Starting Ollama server (/usr/local/bin/ollama)...
✓ Ollama up and running at http://localhost:11434


In [ ]:
import json
import logging
from pathlib import Path

from tqdm.asyncio import tqdm

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.data_structures.memory import Episode
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.memory import JSONMemoryStore, ReflectiveMemory
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool

## Defining the Tool

The same PokéAPI tool used in `recency_memory.ipynb`: look up a Pokémon by name
and return its types and base stats.

In [8]:
def get_pokemon(name: str) -> str:
    """Look up a Pokémon by name and return its types and base stats."""
    url = f"https://pokeapi.co/api/v2/pokemon/{name.lower().strip()}"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "llm-agents-from-scratch/1.0"},
    )
    try:
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        if e.code == 404:  # noqa: PLR2004
            raise ValueError(
                f"Pokémon '{name}' not found. "
                "Check the spelling and try again.",
            ) from e
        raise
    types = [t["type"]["name"] for t in data["types"]]
    stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
    return json.dumps({"name": data["name"], "types": types, "stats": stats})


get_pokemon_tool = SimpleFunctionTool(func=get_pokemon)

In [9]:
STORE_DIR = Path(".")
(STORE_DIR / "reflective_episodes.jsonl").unlink(missing_ok=True)

## Creating the Agent with Memory

`JSONMemoryStore` persists episodes to a newline-delimited JSON file.
`ReflectiveMemory` wraps the store and adds an LLM call at write time: before each
episode is saved, it calls `llm.complete()` with the task and result to produce a
one-sentence lesson, then stores that lesson under
`episode.additional_data["reflection"]`. At recall time the last `n` episodes are
returned, each carrying its reflection. The agent holds the memory in its `memories`
list. `TaskHandler` calls `recall` at task start and `record` at task end
automatically.

In [10]:
store = JSONMemoryStore(
    dir=STORE_DIR,
    filename="reflective_episodes.jsonl",
)
llm = OllamaLLM(model="qwen3:14b", think=False)
memory = ReflectiveMemory(store=store, llm=llm, n=3)
agent = LLMAgent(llm=llm, tools=[get_pokemon_tool], memories=[memory])

## Part 1 — Building Episodic Memory

Four Pokémon lookups: Bulbasaur, Charizard, Snorlax, and Magikarp. Each task calls
`get_pokemon` once and records an episode. After the episode is stored,
`ReflectiveMemory` calls the LLM to distil a one-sentence lesson from the task and
result before persisting.

The first lookup runs with full logging so you can see the tool call and the record
step. The remaining three run concurrently with logging silenced to keep the output
readable.

In [11]:
enable_console_logging(logging.INFO)

task1 = Task(
    instruction=(
        "What are Bulbasaur's types and base stats? "
        "Use the get_pokemon tool. Do not rely on prior knowledge."
    ),
)
result1 = await agent.run(task1)
print(result1)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: What are Bulbasaur's types and base stats? Use the get_pokemon tool. Do not rely on prior knowledge.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: What are Bulbasaur's types and base stats? Use the get_pokemon tool. Do not rely on prior knowledge.
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to call the get_pokemon tool with the name "Bulbasaur" to retrieve its types and base stats.
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Call the get_pokemon tool with the name 'Bulbasaur' to retrieve its types and base stats.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Call the get_pokemon tool with the name 'Bulbasaur' to retrieve its types and base stats.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: get_pokemon
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: {"name": "bulbasaur", "types": ["grass", "poison"], "stats": {"hp": 45, "attack": 49, "defense": 

In [ ]:
logging.disable(logging.INFO)

remaining_tasks = [
    Task(
        instruction=(
            f"What are {name}'s types and base stats? "
            "Use the get_pokemon tool. Do not rely on prior knowledge."
        ),
    )
    for name in ["Charizard", "Snorlax", "Magikarp"]
]

await tqdm.gather(
    *[agent.run(t) for t in remaining_tasks],
    desc="Looking up Pokémon",
)

In [ ]:
logging.disable(logging.NOTSET)

print(await memory.summary())

## The RecencyMemory Baseline

Before examining the reflections, it is worth seeing what `RecencyMemory` would have
surfaced for a new task at this point. A `RecencyMemory` stores no additional data.
The episodes it recalls disclose what happened — the raw JSON stats returned by the
tool and the agent's final answer — but the key insight is not explicitly extracted.
A future agent receiving these episodes would have to re-parse the stat tables to
answer a synthesis question.

In [ ]:
print("What RecencyMemory would recall (same episodes, no reflections):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    ep_without_reflection = Episode(
        task=ep.task,
        rollout=ep.rollout,
        result=ep.result,
        completed_at=ep.completed_at,
    )
    print(str(ep_without_reflection))
    print()

## Part 2 — Inspecting the Reflections

Now look at the same three episodes as stored by `ReflectiveMemory`. Each episode
carries a `<reflection>` tag holding the one-sentence lesson the LLM distilled at
write time. The lessons surface what matters most about each Pokémon — its type
combination, standout stat, or notable weakness — without requiring the agent to
re-read the full stat table.

In [ ]:
print("What ReflectiveMemory recalls (same episodes, with reflections):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    print(str(ep))
    print()

## Part 3 — Synthesis from Recalled Lessons

The synthesis task asks the agent to recommend a Pokémon for a specific battle
scenario using only what it already knows. No tool calls are needed. The agent
receives the three most recent episodes — each with its reflection — in its system
prompt.

The cell below first shows which episodes will be recalled and what reflections they
carry. Then the task runs. Watch the logs: you should see no `🛠️ Executing Tool Call`
lines.

In [ ]:
task5 = Task(
    instruction=(
        "Based on the Pokémon you have already researched, which one would you "
        "least recommend for a competitive battle, and why? "
        "Do not call any tools. Use only what you already know."
    ),
)

print("Episodes that will be recalled for this task:\n")
recalled = await memory.store.read_recent(memory.n)
for ep in recalled:
    print(str(ep))
    print()

In [ ]:
enable_console_logging(logging.INFO)

result5 = await agent.run(task5)
print(result5)

## Key Takeaway

`ReflectiveMemory` converts task experience into transferable lessons. Three concerns
stay cleanly separated: the `LLMAgent` runs tasks; `ReflectiveMemory` distils each
outcome into an explicit one-sentence lesson and delegates storage; `JSONMemoryStore`
persists the enriched episodes.

The comparison in Parts 1 and 2 makes the difference concrete. Raw `RecencyMemory`
episodes surface the full event log — the tool's JSON payload, the agent's formatted
answer — but the lesson is implicit. `ReflectiveMemory` surfaces it in a single
sentence alongside the episode, so a future task can act on it without re-parsing
the full result.

The separation of concerns mirrors `RecencyMemory` and `SimilarityMemory`:
`ReflectiveMemory` decides *what* to generate, *what* to attach, and *how* to format
the recalled context; `JSONMemoryStore` decides *how* to persist and retrieve
episodes. Swapping the store does not affect the reflection logic, and any
`BaseMemoryStore` that implements `read_recent()` works as a drop-in backend.